<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review segmentation

## Set up

In [21]:
# @title Dependencies

# Installation
!pip install strsimpy -q
!pip install pandarallel -q

# First-party installations
import os
from google.colab import drive
from IPython.display import display
import re
import hashlib
import json

# Third-party installations
import pandas as pd
import numpy as np
from tqdm import tqdm
from strsimpy.normalized_levenshtein import NormalizedLevenshtein
from pandarallel import pandarallel

In [22]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_raw")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_processed")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_segmented.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_segmented.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_raw.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed.


In [23]:
# @title Setup definitions

INPUT_DATA = "llm_reviews_results.parquet"

TARGET_SECTION = "weaknesses"

# Define target columns
REVIEW_COL_ORIGINAL = 'llm_review'
REVIEW_COL_EXTRACTED_WEAKNESS = 'extracted_weaknesses'
REVIEW_COL_EXTRACTED_RATING = 'extracted_scores'
REVIEW_COL_SEGMENTS = 'segmented_comment'

# Define id columns
CONFIG_UNI_ID = "config_group"
UNI_ID = "segment_hash"

# Define config columns for grouping hash
config_columns = [
    "model",
    "temperature",
    "reasoning_effort",
    "chain_of_thought",
    "note_taking"
]

# Define the desired order of columns
COL_ORDER = [
    UNI_ID,
    CONFIG_UNI_ID,
    'paper_id',
    'run_signature',
    'model',
    'temperature',
    'reasoning_effort',
    'chain_of_thought',
    'note_taking',
    'llm_notes',
    'llm_review',
    REVIEW_COL_EXTRACTED_WEAKNESS,
    REVIEW_COL_EXTRACTED_RATING,
    REVIEW_COL_SEGMENTS,
    'correctness_rating',
    'technical_novelty_significance_rating',
    'empirical_novelty_significance_rating'
]

# Define keys for numerical ratings
RATING_KEYS = [
    "correctness_rating",
    "technical_novelty_significance_rating",
    "empirical_novelty_significance_rating"
]

# Define specific error strings for review generation/extraction/parsing failures
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"
UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING = "ERROR: UNKNOWN STATE REACHED IN PARALLEL PROCESSING"
EXTRACTION_FAILED_EMPTY_SECTION = "ERROR: EXTRACTION FAILED / SECTION MISSING"

In [24]:
# @title Data import

# Read dataset
df = pd.read_parquet(os.path.join(DATASET_DIR, INPUT_DATA))

# Check and display
display(df.shape)
display(df.head(3))

(9, 9)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using..."
1,vuD2xEtxZcj,gpt-5-nano-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""The paper advocates for ..."
2,vuD2xEtxZcj,gpt-4o-2024-11-20,1.5,None,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{\n ""summary_of_paper"": ""The paper introduces..."


In [25]:
# @title Configuration group identifier

# Create an informative identifier for each configuration group
df[CONFIG_UNI_ID] = df.apply(lambda row: ' | '.join([f'{col}: {row[col]}' for col in config_columns]), axis=1)

# Check and display grouping identifier(s)
display(df.shape)
display(df.head(3))

(9, 10)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...
1,vuD2xEtxZcj,gpt-5-nano-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""The paper advocates for ...",model: gpt-5-nano-2025-08-07 | temperature: na...
2,vuD2xEtxZcj,gpt-4o-2024-11-20,1.5,None,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{\n ""summary_of_paper"": ""The paper introduces...",model: gpt-4o-2024-11-20 | temperature: 1.5 | ...


## Define

In [26]:
# @title Extraction logic

def extract_weaknesses(json_str):
    """
    Parses the json_str and extracts the TARGET_SECTION section.
    Returns a list of strings, or a list containing an error string if parsing fails or the section is missing.
    """
    # Handle specific error strings first
    if json_str in [REVIEW_GENERATION_FAILURE, UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING]:
        return [json_str]

    try:
        # Load columnn data into JSON
        review_data = json.loads(json_str)

        # Isolate the TARGET_SECTION
        target_section = review_data.get(TARGET_SECTION)

        # If target_section is None or empty, return the specific error string
        if not target_section:
            return [EXTRACTION_FAILED_EMPTY_SECTION]

        # Return the whole section
        return target_section

    except Exception as e:
        # Print a hardcoded error message for debugging
        print(f"Warning: Failed to isolate weaknesses. Error: {e}. Review string: {json_str[:100]}...")
        return [EXTRACTION_FAILED_EMPTY_SECTION]

def extract_ratings(json_str, rating_keys):
    """
    Parses the json_str and extracts numerical ratings for specified keys.
    Returns a dictionary of ratings, or None for missing keys.
    If the input is an error string, all ratings will be that error string.
    """
    ratings = {}
    # Handle specific error strings first
    if json_str in [REVIEW_GENERATION_FAILURE, UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING]:
        for key in rating_keys:
            ratings[key] = None # Assign None for numerical rating errors
        return ratings

    try:
        review_data = json.loads(json_str)
        for key in rating_keys:
            ratings[key] = review_data.get(key)
    except Exception as e:
        print(f"Warning: Failed to extract ratings. Error: {e}. Review string: {json_str[:100]}...")
        for key in rating_keys:
            ratings[key] = None # Assign None for numerical rating errors
    return ratings

## Run

In [27]:
# @title Extraction

# Initialize parallel processing for faster extraction
pandarallel.initialize(progress_bar=True)

# Apply the extraction function for weaknesses
df[REVIEW_COL_EXTRACTED_WEAKNESS] = df[REVIEW_COL_ORIGINAL].parallel_apply(extract_weaknesses)

# Apply the extraction function for ratings
extracted_ratings_series = df[REVIEW_COL_ORIGINAL].parallel_apply(lambda x: extract_ratings(x, RATING_KEYS))

# Add the full dictionary of ratings as a new column
df[REVIEW_COL_EXTRACTED_RATING] = extracted_ratings_series

# Expand the dictionary of ratings into separate columns
for key in RATING_KEYS:
    df[key] = extracted_ratings_series.apply(lambda x: x.get(key))

# Check and display extracted section(s) and new rating columns
display(df.shape)
display(df.head())

INFO: Pandarallel will run on 1 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


  "summary_of_paper": "The paper proposes a novel approach to N:M fine-grained sparsity for neural...


  "summary_of_paper": "The paper proposes a novel approach to N:M fine-grained sparsity for neural...


(9, 15)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group,extracted_weaknesses,extracted_scores,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0
1,vuD2xEtxZcj,gpt-5-nano-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""The paper advocates for ...",model: gpt-5-nano-2025-08-07 | temperature: na...,[- Experimental evaluation is constrained by t...,"{'correctness_rating': 4, 'technical_novelty_s...",4.0,4.0,4.0
2,vuD2xEtxZcj,gpt-4o-2024-11-20,1.5,None,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{\n ""summary_of_paper"": ""The paper introduces...",model: gpt-4o-2024-11-20 | temperature: 1.5 | ...,[ERROR: EXTRACTION FAILED / SECTION MISSING],"{'correctness_rating': None, 'technical_novelt...",NaN,NaN,NaN
3,vuD2xEtxZcj,gpt-4o-mini-2024-07-18,1.5,None,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{\n ""summary_of_paper"": ""This paper investiga...",model: gpt-4o-mini-2024-07-18 | temperature: 1...,[The formula derivations are complex and entai...,"{'correctness_rating': 4, 'technical_novelty_s...",4.0,5.0,4.0
4,vuD2xEtxZcj,o4-mini-2025-04-16,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper introduces an...",model: o4-mini-2025-04-16 | temperature: nan |...,[Exact MVUE for 2:4 sparsity is computationall...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0


## Transformation

In [28]:
# @title Long-format

# Create a copy of the extracted section column
df[REVIEW_COL_SEGMENTS] = df[REVIEW_COL_EXTRACTED_WEAKNESS]

# Convert the copied column into long-format
df = df.explode(REVIEW_COL_SEGMENTS)

# Display and check segmented comment(s)
display(df.shape)
display(df.head(3))

(25, 16)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group,extracted_weaknesses,extracted_scores,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,segmented_comment
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0,Experimental details in the notes are summariz...
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0,The optimal 2:4 MVUE is reported to be computa...
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0,Some practical engineering aspects are only di...


In [29]:
# @title Unique identifiers

# Get all columns as basis for unique identifier (hash)
col_for_hash = df.columns.tolist()

# Create a unique identifier (hash)
df[UNI_ID] = df[col_for_hash].astype(str).agg(''.join, axis=1).apply(lambda x: hashlib.sha1(x.encode('utf-8')).hexdigest()[:12])

# Display and check unique identifier(s)
display(df.shape)
display(df.head(3))

(25, 17)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,config_group,extracted_weaknesses,extracted_scores,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,segmented_comment,segment_hash
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0,Experimental details in the notes are summariz...,ab3ce65ba319
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0,The optimal 2:4 MVUE is reported to be computa...,4889f4e5fd66
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",model: gpt-5-mini-2025-08-07 | temperature: na...,[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",5.0,4.0,4.0,Some practical engineering aspects are only di...,e61d79ef0f06


In [30]:
# @title Reorder columns

# Reindex the DataFrame to apply the new column order
df = df.reindex(columns=COL_ORDER)

# Display the head of the DataFrame to show the new column order
display(df.head(3))

,segment_hash,config_group,paper_id,run_signature,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes,llm_review,extracted_weaknesses,extracted_scores,segmented_comment,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating
0,ab3ce65ba319,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,45d644307435,gpt-5-mini-2025-08-07,NaN,low,,Turned off,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",Experimental details in the notes are summariz...,5.0,4.0,4.0
0,4889f4e5fd66,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,45d644307435,gpt-5-mini-2025-08-07,NaN,low,,Turned off,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",The optimal 2:4 MVUE is reported to be computa...,5.0,4.0,4.0
0,e61d79ef0f06,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,45d644307435,gpt-5-mini-2025-08-07,NaN,low,,Turned off,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",[Experimental details in the notes are summari...,"{'correctness_rating': 5, 'technical_novelty_s...",Some practical engineering aspects are only di...,5.0,4.0,4.0


In [31]:
# @title Results

# Save the long-format DataFrame to JSONL
df.to_json(RESULTS_JSON_PATH, orient='records', lines=True)
print(f"DataFrame saved to JSONL at: {RESULTS_JSON_PATH}")

# Save the long-format DataFrame to Parquet
df.to_parquet(RESULTS_PATH, index=False)
print(f"DataFrame saved to Parquet at: {RESULTS_PATH}")

DataFrame saved to JSONL at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed/llm_reviews_segmented.jsonl
DataFrame saved to Parquet at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed/llm_reviews_segmented.parquet
